# Generowanie i interpretacja reguł eksperckich dla klasyfikacji filmów Netflix z wykorzystaniem ML i LLM

**Autorzy:** Stas Kochevenko, Diana Hunchak  
**Temat:** Preprocessing, oczyszczanie i przygotowanie zbioru danych (Feature Engineering) pod kątem klasyfikacji oraz ekstrakcji reguł decyzyjnych.

### Cel projektu
Stworzenie zaawansowanego systemu generującego interpretowalne reguły eksperckie na podstawie danych o filmach platformy Netflix z wykorzystaniem algorytmów uczenia maszynowego (Machine Learning) oraz modeli językowych (LLM). System klasyfikuje filmy jako **sukces** lub **porażkę**, a następnie automatycznie generuje i interpretuje reguły warunkowe typu `IF-THEN` na naturalny język biznesowy.



## Preprocessing:
1. **Inicjalizacja i ładowanie danych:** Zaimportowanie niezbędnych bibliotek (`pandas`, `numpy`, `matplotlib`) oraz wczytanie bazy danych `netflix_movies_detailed_up_to_2025.csv`.
2. **Czyszczenie danych (Usuwanie szumu):** Eliminacja filmów bez ocen (`vote_average = 0`) oraz odfiltrowanie produkcji o zbyt niskiej liczbie głosów (`vote_count`), aby zapewnić reprezentatywność statystyczną danych.
3. **Selekcja cech (Feature Selection):** Usunięcie kolumn nieprzydatnych dla modelu drzewa decyzyjnego (unikalne identyfikatory, surowe opisy tekstowe, linki oraz cechy o zbyt wysokiej kardynalności, jak pełna obsada czy reżyser).
4. **Obsługa brakujących wartości (Missing Values):** Imputacja danych dla zmiennych kategorycznych (wstawienie etykiety `'Unknown'`) oraz numerycznych (uzupełnienie medianą).
5. **Inżynieria cech (Feature Engineering):** Definiowanie binarnej zmiennej docelowej (`is_success`) oraz uproszczenie struktury gatunków (`genres`) do głównej kategorii (Main Genre) w celu ułatwienia późniejszej interpretacji reguł przez LLM.

In [2]:
# Krok 1: Import wymaganych bibliotek i interaktywne ładowanie zbioru danych
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io

# Interaktywny upload pliku (dedykowane dla środowiska Google Colab)
try:
    from google.colab import files
    print("Proszę wgrać plik CSV z danymi Netflixa (np. netflix_movies_detailed_up_to_2025.csv):")
    uploaded = files.upload()

    # Pobranie nazwy wgranego pliku
    file_name = list(uploaded.keys())[0]

    # Wczytanie danych z wgranego pliku do obiektu DataFrame
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
    print(f"\nPomyślnie wczytano plik: {file_name}")

except ImportError:
    # Fallback dla lokalnego środowiska Jupyter Notebook (jeśli kod nie jest uruchamiany w Colabie)
    print("Środowisko lokalne wykryte. Wczytywanie pliku z dysku...")
    file_name = 'netflix_movies_detailed_up_to_2025.csv'
    df = pd.read_csv(file_name)
    print(f"Pomyślnie wczytano plik: {file_name}")

print(f"Początkowy rozmiar zbioru danych: {df.shape}")
display(df.head(3))

Proszę wgrać plik CSV z danymi Netflixa (np. netflix_movies_detailed_up_to_2025.csv):


Saving netflix_movies_detailed_up_to_2025.csv to netflix_movies_detailed_up_to_2025.csv

Pomyślnie wczytano plik: netflix_movies_detailed_up_to_2025.csv
Początkowy rozmiar zbioru danych: (16000, 18)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868


## Krok 2: Oczyszczanie danych i filtrowanie szumu informacyjnego
Zgodnie z metodologią projektu, odrzucamy filmy, które nie posiadają żadnych ocen (`vote_average = 0`). Ponadto wprowadzamy filtr dla minimalnej liczby głosów (`vote_count >= 10`). Zapobiega to sytuacji, w której film z jedną maksymalną oceną (np. 10/10) jest błędnie traktowany przez model jako globalny sukces, co mogłoby zniekształcić reguły eksperckie.

In [3]:
# Filtrowanie filmów na podstawie ocen oraz minimalnej liczby głosów
df_filtered = df[df['vote_average'] > 0].copy()
df_filtered = df_filtered[df_filtered['vote_count'] >= 10]

print(f"Rozmiar zbioru po odfiltrowaniu szumu: {df_filtered.shape}")

Rozmiar zbioru po odfiltrowaniu szumu: (14183, 18)


## Krok 3: Selekcja cech (Feature Selection)
Usuwamy kolumny o charakterze ściśle unikalnym lub tekstowym (`show_id`, `title`, `description`), ponieważ klasyfikator drzewa decyzyjnego (Decision Tree) nie jest w stanie wyekstrahować z nich ogólnych reguł strukturalnych bez zaawansowanego przetwarzania NLP. Usuwamy również kolumny `cast` oraz `director` z powodu zbyt wysokiej liczby unikalnych wartości (kardynalności), co mogłoby prowadzić do przeuczenia modelu (overfitting). Skupiamy się na roku wydania, usuwając `date_added`.

In [4]:
# Lista kolumn rekomendowanych do usunięcia w celu redukcji wymiarowości
columns_to_drop = ['show_id', 'type', 'title', 'description', 'date_added', 'cast', 'director']

df_selected = df_filtered.drop(columns=columns_to_drop, errors='ignore')
print(f"Cechy pozostałe w zbiorze do dalszej analizy: {df_selected.columns.tolist()}")

Cechy pozostałe w zbiorze do dalszej analizy: ['country', 'release_year', 'rating', 'duration', 'genres', 'language', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']


## Krok 4: Zaawansowana obsługa brakujących wartości (Missing Values)
Analizujemy procent brakujących danych. Kolumny z krytycznym brakiem danych (>50%, np. całkowicie pusta kolumna `duration`) są automatycznie usuwane. Dla pozostałych zmiennych kategorycznych stosujemy bezpieczną imputację etykietą `'Unknown'`.

In [7]:
# 1. Analiza braków
missing_percent = (df_selected.isnull().sum() / len(df_selected)) * 100
print("Procent brakujących danych przed czyszczeniem:")
print(round(missing_percent, 2).astype(str) + '%')

# 2. Usuwanie kolumn z brakiem > 50% danych (rozwiąże problem z 'duration')
THRESHOLD = 50.0
cols_to_drop = missing_percent[missing_percent > THRESHOLD].index.tolist()

if cols_to_drop:
    print(f"\n[Akcja] Usuwam kolumny z powodu braku większości danych (>50%): {cols_to_drop}")
    df_selected = df_selected.drop(columns=cols_to_drop)

# 3. Imputacja pozostałych braków kategorycznych
categorical_cols = ['country', 'rating', 'language', 'genres']
existing_cat_cols = [col for col in categorical_cols if col in df_selected.columns]

if existing_cat_cols:
    df_selected[existing_cat_cols] = df_selected[existing_cat_cols].fillna('Unknown')

print("\nLiczba brakujących wartości po inteligentnym czyszczeniu:\n", df_selected.isnull().sum())

Procent brakujących danych przed czyszczeniem:
country           0.0%
release_year      0.0%
rating            0.0%
duration        100.0%
genres            0.0%
language          0.0%
popularity        0.0%
vote_count        0.0%
vote_average      0.0%
budget            0.0%
revenue           0.0%
dtype: object

[Akcja] Usuwam kolumny z powodu braku większości danych (>50%): ['duration']

Liczba brakujących wartości po inteligentnym czyszczeniu:
 country         0
release_year    0
rating          0
genres          0
language        0
popularity      0
vote_count      0
vote_average    0
budget          0
revenue         0
dtype: int64


## Krok 5: Inżynieria cech i definicja zmiennej docelowej (Target Variable)
Tworzymy binarną zmienną docelową `is_success` (próg oceny 6.5). Upraszczamy gatunki do `main_genre`, aby reguły były czytelniejsze dla LLM. Na koniec usuwamy cechy powodujące wyciek danych (Data Leakage) – `vote_average`, `vote_count` oraz `popularity`.

In [8]:
# 1. Definicja Target Variable
SUCCESS_THRESHOLD = 6.5
df_selected['is_success'] = (df_selected['vote_average'] >= SUCCESS_THRESHOLD).astype(int)

# 2. Ekstrakcja głównego gatunku
if 'genres' in df_selected.columns:
    df_selected['main_genre'] = df_selected['genres'].apply(lambda x: x.split(',')[0] if isinstance(x, str) else x)
    df_selected = df_selected.drop(columns=['genres'])

# 3. Usunięcie Data Leakage
features_final = df_selected.drop(columns=['vote_average', 'vote_count', 'popularity'], errors='ignore')

print("Gotowy zbiór danych do budowy modelu (Decision Tree Classifier):")
display(features_final.head())
print(f"\nOstateczny kształt danych: {features_final.shape}")

Gotowy zbiór danych do budowy modelu (Decision Tree Classifier):


,country,release_year,rating,language,budget,revenue,is_success,main_genre
0,United States of America,2010,6.380,en,165000000,752600867,0,Comedy
1,"United Kingdom, United States of America",2010,8.369,en,160000000,839030630,1,Action
2,"United Kingdom, United States of America",2010,7.744,en,250000000,954305868,1,Adventure
3,United States of America,2010,7.600,en,260000000,592461732,1,Animation
4,United States of America,2010,7.800,en,165000000,494879471,1,Fantasy



Ostateczny kształt danych: (14183, 8)
